In [58]:
!pip install earthengine-api geemap -q

In [59]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='mrv-poc-500917')
print("Connected ✅")

Connected ✅


In [60]:
coords = [
    [75.7583337, 13.4004903],[75.7581502, 13.3995258],
    [75.7593232, 13.3995341],[75.7594960, 13.4003524],
    [75.7583337, 13.4004903],
]
aoi = ee.Geometry.Polygon([coords])
area_ha = aoi.area().getInfo()/10000
print(f"Plot area: {area_ha:.2f} ha")

Plot area: 1.27 ha


In [61]:
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi).filterDate('2024-11-01','2025-03-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20))
      .median().clip(aoi))
ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI')
mean_ndvi = ndvi.reduceRegion(ee.Reducer.mean(), aoi, 10).get('NDVI').getInfo()
print(f"Mean NDVI: {mean_ndvi:.3f}")

m = geemap.Map(); m.centerObject(aoi,16)
m.addLayer(s2,{'bands':['B4','B3','B2'],'min':0,'max':3000},'True colour')
m.addLayer(ndvi,{'min':0,'max':0.8,'palette':['white','khaki','green','darkgreen']},'NDVI')
m

Mean NDVI: 0.873


Map(center=[13.399977694910868, 75.75881015977178], controls=(WidgetControl(options=['position', 'transparent_…

In [62]:
ARECA = {
    'scientific_name': 'Areca catechu',
    'equation': 'Y = 0.03883 * H * D^1.2',     # combined, R2=0.96
    'coef_k': 0.03883, 'coef_d_exp': 1.2,
    'source': 'Prayogo et al. 2018, AGRIVITA 40(3):381-389, DOI 10.17503/agrivita.v40i3.1124',
    'region_note': 'Indonesia-derived; prefer India eqn if found',
    'carbon_fraction': 0.47,                    # IPCC 2006 default
    'rsr_provisional': 0.24,                    # IPCC tropical default — PALM PROXY, flag in limitations
    'rsr_source': 'IPCC 2006 GL Vol.4 (tree-derived; no areca-specific value found)',
}

In [63]:
def areca_agb_kg(D_cm, H_m):
    return ARECA['coef_k'] * H_m * (D_cm ** ARECA['coef_d_exp'])   # kg/plant

def to_co2e_kg(biomass_kg):
    return biomass_kg * ARECA['carbon_fraction'] * 3.67

def stand_co2e(D_list, H_list):
    total = 0.0
    for D, H in zip(D_list, H_list):
        agb = areca_agb_kg(D, H)
        bgb = agb * ARECA['rsr_provisional']     # provisional palm BGB
        total += to_co2e_kg(agb + bgb)
    return total / 1000                          # tCO2e

In [64]:
n_palms = 60                                     # assumed; field-counted later
D_y0 = [13.0]*n_palms; H_y0 = [9.0]*n_palms      # assumed baseline (cm, m)
D_y1 = [13.2]*n_palms; H_y1 = [9.6]*n_palms      # assumed after 1 yr (palms grow in HEIGHT)
agb_bgb_increment = stand_co2e(D_y1, H_y1) - stand_co2e(D_y0, H_y0)
print(f"Annual AGB+BGB increment: {agb_bgb_increment:.3f} tCO2e")

Annual AGB+BGB increment: 0.084 tCO2e


In [65]:
soc = ee.Image('projects/soilgrids-isric/soc_mean').select('soc_0-5cm_mean').clip(aoi)
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)
slope = ee.Terrain.slope(dem)
landcover = ee.Image('ESA/WorldCover/v200/2021').clip(aoi)
strata = landcover.divide(10).floor().multiply(10).add(slope.gt(10).multiply(1))

mean_soc = soc.reduceRegion(ee.Reducer.mean(), aoi, 250).get('soc_0-5cm_mean').getInfo()
print(f"Baseline SOC concentration (0-5cm): {mean_soc/10:.1f} g/kg ({mean_soc/100:.2f}%)")
print("NOTE: concentration, not stock; stock needs bulk density x depth (field data)")

m2 = geemap.Map(); m2.centerObject(aoi,16)
m2.addLayer(soc,{'min':0,'max':400,'palette':['tan','brown','black']},'SOC baseline')
m2.addLayer(strata.randomVisualizer(),{},'Strata')
m2

Baseline SOC concentration (0-5cm): 63.5 g/kg (6.35%)
NOTE: concentration, not stock; stock needs bulk density x depth (field data)


Map(center=[13.399977694910868, 75.75881015977178], controls=(WidgetControl(options=['position', 'transparent_…

In [66]:
region = aoi.centroid().buffer(6000)

# --- REAL forcing data (these go in the paper) ---
rain = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
        .filterDate('2024-01-01','2024-12-31').sum())
rain_mm = rain.reduceRegion(ee.Reducer.mean(), region, 5500).getInfo().get('precipitation') or 1879.0

temp = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
        .filterDate('2024-01-01','2024-12-31').select('temperature_2m').mean())
temp_k = temp.reduceRegion(ee.Reducer.mean(), region, 11000).getInfo().get('temperature_2m') or 294.7
temp_c = temp_k - 273.15
print(f"Forcing (real): rain={rain_mm:.0f} mm, temp={temp_c:.1f} C")

# --- SOC change: conservative literature-based rate (POC stand-in) ---
# Agroforestry SOC accrual in the literature ~0.2-0.5 tC/ha/yr; conservative end used.
# FLAGGED: replace with field-calibrated RothC before any credit issuance.
soc_change_tC = 0.3                        # tC/ha/yr, conservative
soc_change_tco2e = soc_change_tC * 3.67    # tCO2e/ha/yr
print(f"SOC change (literature-based, conservative): {soc_change_tco2e:.3f} tCO2e/ha/yr")

Forcing (real): rain=1879 mm, temp=21.6 C
SOC change (literature-based, conservative): 1.101 tCO2e/ha/yr


In [67]:
def net_credits(agb_bgb_t, soc_t, leakage=0.07, uncertainty=0.20):
    gross  = agb_bgb_t + soc_t
    after  = gross * (1 - leakage)
    issued = after * (1 - uncertainty)
    return {'gross':gross, 'after_leakage':after,
            'issued_lower':issued, 'buffer_held':after-issued}

result = net_credits(agb_bgb_increment, soc_change_tco2e * area_ha)
print(result)

{'gross': 1.477754983925038, 'after_leakage': 1.3743121350502852, 'issued_lower': 1.0994497080402281, 'buffer_held': 0.27486242701005703}


In [68]:
import json, hashlib
def hash_chain(records):
    prev = "0"*64; out=[]
    for r in records:
        h = hashlib.sha256((json.dumps(r,sort_keys=True)+prev).encode()).hexdigest()
        out.append({**r,'prev':prev,'hash':h}); prev=h
    return out

log = [
    {'day':1, 'mean_ndvi':round(mean_ndvi,3)},
    {'day':2, 'increment_tco2e':round(agb_bgb_increment,3)},
    {'day':5, 'issued_tco2e':round(result['issued_lower'],3)},
]
for row in hash_chain(log):
    print(row.get('day'), row['hash'][:16], '…')

1 b3bd5d35c45d121e …
2 5bb7414623197789 …
5 18d0de8e3919c64a …


In [69]:
print("=== MRV POC SUMMARY ===")
print(f"Plot: {area_ha:.2f} ha | Mean NDVI: {mean_ndvi:.3f}")
print(f"AGB+BGB increment: {agb_bgb_increment:.3f} tCO2e")
print(f"Issued (conservative): {result['issued_lower']:.3f} tCO2e | buffer: {result['buffer_held']:.3f}")
print("\nSensitivity — uncertainty deduction vs issued credits:")
for u in [0.30,0.25,0.20,0.15,0.10]:
    r = net_credits(agb_bgb_increment, soc_change_tco2e*area_ha, uncertainty=u)
    print(f"  uncertainty {int(u*100)}% -> issued {r['issued_lower']:.3f} tCO2e")

=== MRV POC SUMMARY ===
Plot: 1.27 ha | Mean NDVI: 0.873
AGB+BGB increment: 0.084 tCO2e
Issued (conservative): 1.099 tCO2e | buffer: 0.275

Sensitivity — uncertainty deduction vs issued credits:
  uncertainty 30% -> issued 0.962 tCO2e
  uncertainty 25% -> issued 1.031 tCO2e
  uncertainty 20% -> issued 1.099 tCO2e
  uncertainty 15% -> issued 1.168 tCO2e
  uncertainty 10% -> issued 1.237 tCO2e
